# 15 · MLWA Journal Verified-Only Reports, Coverage Audit & Publication Figures

> **Standalone Single-File Notebook**: All required modules are embedded directly below. Upload only this `.ipynb` file to Google Colab and run all cells (`Runtime -> Run all`); no repository clone or zip bundle is required.

In [ ]:
NOTEBOOK_CODE_SHA256 = '19323ff8bddabf8609c0a0241a3255a6ae3c4d022c9306bc6ab7fe61fbcd789b'

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB: bool = (
    'COLAB_RELEASE_TAG' in os.environ
    or 'COLAB_BACKEND_VERSION' in os.environ
    or 'google.colab' in sys.modules
)


def _running_in_notebook() -> bool:
    if 'ipykernel' not in sys.modules and 'google.colab' not in sys.modules:
        return False
    try:
        from IPython import get_ipython
        ip = get_ipython()
        return ip is not None and ip.__class__.__name__ != 'TerminalInteractiveShell'
    except Exception:
        return False


def _setup_colab_environment() -> Path:
    if IN_COLAB:
        import importlib.util
        import subprocess
        deps = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'sklearn': 'scikit-learn', 'pyarrow': 'pyarrow', 'matplotlib': 'matplotlib'}
        missing = [pkg for mod, pkg in deps.items() if importlib.util.find_spec(mod) is None]
        if missing:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
        if not Path('/content/drive/MyDrive').is_dir():
            from google.colab import drive
            drive.mount('/content/drive')
    return Path(os.environ.get('DR_ROOT', '/content/drive/MyDrive/dr_thesis' if IN_COLAB else Path.cwd())).resolve()


if __name__ == '__main__' and _running_in_notebook():
    _REPO = _setup_colab_environment()
    P0_DIR = None
    E0_DIR = None
    E1_DIR = None
    JOURNAL_RUNS_DIR = None
    OUTPUT_DIR = None
    SMOKE = True

## Embedded Standalone Implementation (3 self-contained modules)

### Module `experiments.mlwa.constants`

In [ ]:
"""Canonical constants, status codes, and portable environment resolution for MLWA journal experiments."""


import os
from pathlib import Path
from typing import Dict, Optional, Tuple

COHORT_ORDER: Tuple[str, ...] = ("aptos", "idrid", "messidor2", "ddr", "eyepacs", "deepdrid")
CANONICAL_FOLD_TO_COHORT: Dict[int, str] = {i: c for i, c in enumerate(COHORT_ORDER)}
COHORT_TO_FOLD: Dict[str, int] = {c: i for i, c in enumerate(COHORT_ORDER)}

COHORT_LABELS: Dict[str, str] = {
    "aptos": "APTOS 2019",
    "idrid": "IDRiD",
    "messidor2": "Messidor-2",
    "ddr": "DDR",
    "eyepacs": "EyePACS",
    "deepdrid": "DeepDRiD",
}

PROB_COLS: Tuple[str, ...] = ("p0", "p1", "p2", "p3", "p4")
THETA_COLS: Tuple[str, ...] = ("theta0", "theta1", "theta2", "theta3")
TIE_TOL: float = 1e-12
SCHEMA_VERSION: str = "mlwa_journal_v1"

STATUS_PLANNED = "PLANNED"
STATUS_PREFLIGHT_OK = "PREFLIGHT_OK"
STATUS_PENDING_ASSETS = "PENDING_ASSETS"
STATUS_RUNNING = "RUNNING"
STATUS_INTERRUPTED = "INTERRUPTED"
STATUS_FAILED = "FAILED"
STATUS_COMPLETE = "COMPLETE"
STATUS_DIAGNOSTIC_REPLAY = "DIAGNOSTIC_REPLAY"
STATUS_SKIPPED_GATE_NOT_MET = "SKIPPED_GATE_NOT_MET"
STATUS_TEST_ONLY = "TEST_ONLY"

STATUS_CODES: Tuple[str, ...] = (
    STATUS_PLANNED,
    STATUS_PREFLIGHT_OK,
    STATUS_PENDING_ASSETS,
    STATUS_RUNNING,
    STATUS_INTERRUPTED,
    STATUS_FAILED,
    STATUS_COMPLETE,
    STATUS_DIAGNOSTIC_REPLAY,
    STATUS_SKIPPED_GATE_NOT_MET,
    STATUS_TEST_ONLY,
)

REQUIRED_PREDICTION_COLUMNS: Tuple[str, ...] = (
    "study_id",
    "protocol_id",
    "run_id",
    "experiment",
    "config_id",
    "architecture_id",
    "head_type",
    "objective_id",
    "clm_link",
    "fold_id",
    "cohort",
    "seed_train",
    "split_role",
    "epoch",
    "checkpoint_sha256",
    "image_key",
    "image_id",
    "patient_key",
    "patient_identity_status",
    "y_true",
    *PROB_COLS,
    "pred_native",
    "s",
    *THETA_COLS,
    "feature_file",
    "feature_row",
    "manifest_sha256",
    "preprocessing_id",
    "schema_version",
    "code_sha256",
    "plan_sha256",
)


def find_repo_root(start: Optional[Path] = None) -> Path:
    """Locate repository root cleanly without hardcoded external Windows paths."""
    cands = []
    if start is not None:
        cands.append(Path(start).resolve())
    cands.append(Path.cwd().resolve())
    if "__file__" in globals():
        cands.append(Path(__file__).resolve().parents[2])
    for base in cands:
        for cur in (base, *base.parents):
            if (cur / "pyproject.toml").is_file() and (cur / "experiments" / "mlwa").is_dir():
                return cur
    if "__file__" in globals():
        return Path(__file__).resolve().parents[2]
    return Path.cwd().resolve()


def resolve_runtime_paths(repo_root: Optional[Path] = None) -> Dict[str, Path]:
    """Resolve portable DR_* paths for local and Google Colab sessions."""
    root = find_repo_root(repo_root)
    dr_root = Path(os.environ.get("DR_ROOT", str(root))).expanduser().resolve()
    inputs = Path(os.environ.get("DR_INPUTS", str(dr_root / "inputs"))).expanduser().resolve()
    outputs_base = Path(os.environ.get("DR_JOURNAL_OUTPUTS", os.environ.get("DR_OUTPUTS", str(dr_root / "outputs")))).expanduser().resolve()
    work = Path(os.environ.get("DR_WORK", str(dr_root / "work"))).expanduser().resolve()

    images = Path(os.environ.get("DR_IMAGES", str(inputs / "native_cropped"))).expanduser().resolve()
    folds = Path(os.environ.get("DR_FOLDS", str(inputs / "lodo_folds"))).expanduser().resolve()
    checkpoints = Path(os.environ.get("DR_CHECKPOINTS", str(inputs / "checkpoints"))).expanduser().resolve()
    pooled_ckpts = Path(os.environ.get("DR_POOLED_CHECKPOINTS", str(inputs / "checkpoints_head_ablation"))).expanduser().resolve()
    pooled_meta = Path(os.environ.get("DR_POOLED_METADATA", str(inputs / "hybrid"))).expanduser().resolve()
    p0_run = Path(os.environ.get("DR_P0_RUN", str(root / "result" / "p0_fp32_main_lodo_v1"))).expanduser().resolve()

    return {
        "repo_root": root,
        "dr_root": dr_root,
        "inputs": inputs,
        "outputs": outputs_base,
        "work": work,
        "images": images,
        "folds": folds,
        "checkpoints": checkpoints,
        "pooled_checkpoints": pooled_ckpts,
        "pooled_metadata": pooled_meta,
        "p0_run": p0_run,
    }

### Module `experiments.mlwa.data_contract`

In [ ]:
"""Data, manifest, image-identity, P0 cache, and prediction schema contracts."""


import hashlib
import json
import os
import shutil
import zipfile
from pathlib import Path, PurePosixPath
from typing import Any, Dict, List, Optional, Sequence, Tuple
from uuid import uuid4

import cv2
import numpy as np
import pandas as pd


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def object_hash(value: Any) -> str:
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    ).hexdigest()


def atomic_write_json(path: Path, value: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f"{path.name}.{uuid4().hex[:8]}.tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")
    os.replace(tmp, path)


def relative_image_path(value: str) -> str:
    parts = PurePosixPath(str(value).replace("\\", "/")).parts
    lower = [p.lower() for p in parts]
    if "native_cropped" in lower:
        parts = parts[lower.index("native_cropped") + 1 :]
    if not parts or ".." in parts or any(":" in p for p in parts) or parts[0] == "/":
        raise ValueError(f"Unsafe or unresolved relative image path: {value}")
    return PurePosixPath(*parts).as_posix()


def stage_archive_safe(source: Path, workspace: Path) -> Path:
    """Safely extract an immutable ZIP to a keyed local folder or return an existing directory."""
    source = Path(source).expanduser().resolve(strict=True)
    if source.is_dir():
        return source
    with zipfile.ZipFile(source) as archive:
        entries = archive.infolist()
        seen = set()
        for info in entries:
            name = info.filename.replace("\\", "/")
            rel = PurePosixPath(name)
            is_symlink = ((info.external_attr >> 16) & 0o170000) == 0o120000
            if rel.is_absolute() or ".." in rel.parts or ":" in name or is_symlink or not rel.parts:
                raise ValueError(f"Unsafe ZIP member: {info.filename}")
            if name.casefold() in seen:
                raise ValueError(f"Duplicate ZIP member path: {info.filename}")
            seen.add(name.casefold())

        index = [(m.filename, m.file_size, m.CRC) for m in entries]
        key = object_hash(index)
        target = Path(workspace) / f"{source.stem}_{key[:16]}"
        ready = target / ".archive-complete"
        if ready.is_file() and ready.read_text(encoding="utf-8").strip() == key:
            return target

        temp = target.with_name(f"{target.name}.{uuid4().hex}.partial")
        temp.mkdir(parents=True, exist_ok=True)
        try:
            for m in entries:
                parts = PurePosixPath(m.filename.replace("\\", "/")).parts
                dest = temp.joinpath(*parts)
                if not dest.resolve().is_relative_to(temp.resolve()):
                    raise ValueError(f"Path traversal blocked: {m.filename}")
                if m.is_dir():
                    dest.mkdir(parents=True, exist_ok=True)
                else:
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(m) as src, dest.open("wb") as dst:
                        shutil.copyfileobj(src, dst, 2**20)
            (temp / ".archive-complete").write_text(key, encoding="utf-8")
            try:
                temp.rename(target)
            except OSError:
                if not ready.is_file() or ready.read_text(encoding="utf-8").strip() != key:
                    raise
        finally:
            if temp.exists():
                shutil.rmtree(temp)
    return target


def normalize_manifest_df(
    df: pd.DataFrame,
    patient_map: Optional[Path] = None,
    images_root: Optional[Path] = None,
) -> pd.DataFrame:
    """Attach canonical image_key, grouping/patient_key, and identity status cleanly."""
    out = df.copy()
    id_col = "image_id"
    if id_col not in out.columns:
        raise ValueError("Manifest missing required column 'image_id'.")
    y_col = "diagnosis" if "diagnosis" in out.columns else ("y_true" if "y_true" in out.columns else None)
    if y_col is None:
        raise ValueError("Manifest missing label column ('diagnosis' or 'y_true').")
    cohort_col = "dataset_key" if "dataset_key" in out.columns else ("cohort" if "cohort" in out.columns else None)
    if cohort_col is None:
        raise ValueError("Manifest missing cohort column ('dataset_key' or 'cohort').")

    out["image_id"] = out[id_col].astype(str)
    out["cohort"] = (
        out[cohort_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({"messidor-2": "messidor2", "messidor_2": "messidor2", "deep-drid": "deepdrid", "deep_drid": "deepdrid"})
    )
    out["y_true"] = pd.to_numeric(out[y_col], errors="raise").astype(int)
    if not out["y_true"].isin(range(5)).all():
        raise ValueError("Manifest contains invalid grade outside {0..4}.")

    if "image_key" not in out.columns:
        out["image_key"] = out["cohort"] + "::" + out["image_id"]
    else:
        out["image_key"] = out["image_key"].astype(str)

    if out["image_key"].duplicated().any():
        dups = out.loc[out["image_key"].duplicated(), "image_key"].head(5).tolist()
        raise ValueError(f"Duplicate image_key entries in manifest: {dups}")

    if "patient_key" in out.columns:
        out["patient_key"] = out["patient_key"].astype(str)
    elif "patient_id" in out.columns:
        out["patient_key"] = out["patient_id"].astype(str)
    else:
        out["patient_key"] = out["image_key"]

    if "patient_identity_status" not in out.columns:
        out["patient_identity_status"] = "legacy_unverified"

    if patient_map is not None:
        mapping = pd.read_csv(patient_map, dtype=str)
        if mapping["image_key"].duplicated().any() or mapping["patient_key"].isna().any():
            raise ValueError(f"Invalid verified patient mapping: {patient_map}")
        mp = mapping.set_index("image_key")["patient_key"]
        known = out["image_key"].isin(mp.index)
        out.loc[known, "patient_key"] = out.loc[known, "image_key"].map(mp)
        out.loc[known, "patient_identity_status"] = "verified"

    path_src_col = next((c for c in ("native_path", "raw_full_path", "original_relative_path") if c in out.columns), None)
    if path_src_col is not None:
        out["original_relative_path"] = out[path_src_col].astype(str).map(relative_image_path)
    if images_root is not None and "original_relative_path" in out.columns:
        root = Path(images_root)
        out["runtime_path"] = [str(root.joinpath(*PurePosixPath(p).parts)) for p in out["original_relative_path"]]
    return out


def verify_images_readable(
    df: pd.DataFrame,
    check_decode: bool = True,
    max_checks: Optional[int] = None,
) -> Dict[str, Any]:
    """Check file presence and OpenCV decode health; fail explicitly rather than skipping silently."""
    if "runtime_path" not in df.columns:
        raise ValueError("runtime_path column required to verify image files.")
    paths = df["runtime_path"].astype(str).tolist()
    if max_checks is not None:
        paths = paths[:max_checks]

    missing: List[str] = []
    unreadable: List[str] = []
    for p in paths:
        fp = Path(p)
        if not fp.is_file():
            missing.append(p)
            continue
        if check_decode:
            img = cv2.imread(str(fp))
            if img is None or img.size == 0:
                unreadable.append(p)

    if missing:
        raise FileNotFoundError(f"{len(missing)} missing images; first 3: {missing[:3]}")
    if unreadable:
        raise OSError(f"{len(unreadable)} unreadable/corrupt images; first 3: {unreadable[:3]}")
    return {"checked": len(paths), "missing": 0, "unreadable": 0}


def build_natural_source_train(
    folds_dir: Path,
    fold_id: int,
    patient_map: Optional[Path] = None,
    images_root: Optional[Path] = None,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Reconstruct natural-prevalence 85% source_train for fold_id from non-target test.csv - val.csv."""
    folds_dir = Path(folds_dir)
    if fold_id not in CANONICAL_FOLD_TO_COHORT:
        raise ValueError(f"Invalid fold_id={fold_id}")
    held_cohort = CANONICAL_FOLD_TO_COHORT[fold_id]

    va_raw = pd.read_csv(folds_dir / f"fold_{fold_id}_{held_cohort}_val.csv", dtype=str)
    va_df = normalize_manifest_df(va_raw, patient_map=patient_map, images_root=images_root)

    source_parts: List[pd.DataFrame] = []
    input_hashes: Dict[str, str] = {
        f"fold_{fold_id}_{held_cohort}_val.csv": sha256_file(folds_dir / f"fold_{fold_id}_{held_cohort}_val.csv")
    }
    for src_fold, src_cohort in CANONICAL_FOLD_TO_COHORT.items():
        if src_fold == fold_id:
            continue
        tp = folds_dir / f"fold_{src_fold}_{src_cohort}_test.csv"
        input_hashes[tp.name] = sha256_file(tp)
        part = normalize_manifest_df(pd.read_csv(tp, dtype=str), patient_map=patient_map, images_root=images_root)
        source_parts.append(part)

    pool_df = pd.concat(source_parts, ignore_index=True)
    if pool_df["image_key"].duplicated().any():
        raise ValueError(f"Duplicate image_key detected across source cohorts for fold {fold_id}.")

    va_keys = set(va_df["image_key"])
    nat_train = pool_df[~pool_df["image_key"].isin(va_keys)].copy().reset_index(drop=True)

    # Verify zero group overlap between reconstructed natural source_train and source_val
    nat_groups = set(nat_train["patient_key"])
    va_groups = set(va_df["patient_key"])
    overlap = nat_groups & va_groups
    if overlap:
        raise ValueError(f"Fold {fold_id} grouping-key overlap between natural source_train and val: {len(overlap)}")

    nat_counts = {int(k): int((nat_train["y_true"] == k).sum()) for k in range(5)}
    meta = {
        "fold_id": fold_id,
        "held_out_cohort": held_cohort,
        "pool_images": int(len(pool_df)),
        "natural_source_train_images": int(len(nat_train)),
        "source_val_images": int(len(va_df)),
        "realized_train_ratio": float(len(nat_train) / len(pool_df)),
        "natural_grade_counts": nat_counts,
        "input_manifest_sha256": input_hashes,
        "reconstructed_manifest_sha256": object_hash(
            {"image_keys": nat_train["image_key"].tolist(), "y_true": nat_train["y_true"].tolist()}
        ),
    }
    return nat_train, meta


def audit_lodo_manifests(
    folds_dir: Path,
    images_root: Optional[Path] = None,
    check_decode: bool = False,
) -> Dict[str, Any]:
    """Comprehensive audit of LODO CSV manifests, group leakage, quota vs natural counts, and optional images."""
    folds_dir = Path(folds_dir)
    fold_reports: List[Dict[str, Any]] = []
    inventory_per_cohort: Dict[str, int] = {}

    for fold_id, cohort in CANONICAL_FOLD_TO_COHORT.items():
        p_tr = folds_dir / f"fold_{fold_id}_{cohort}_train.csv"
        p_va = folds_dir / f"fold_{fold_id}_{cohort}_val.csv"
        p_te = folds_dir / f"fold_{fold_id}_{cohort}_test.csv"
        for p in (p_tr, p_va, p_te):
            if not p.is_file():
                raise FileNotFoundError(f"Missing LODO manifest: {p}")

        tr = normalize_manifest_df(pd.read_csv(p_tr, dtype=str), images_root=images_root)
        va = normalize_manifest_df(pd.read_csv(p_va, dtype=str), images_root=images_root)
        te = normalize_manifest_df(pd.read_csv(p_te, dtype=str), images_root=images_root)

        if set(te["cohort"].unique()) != {cohort}:
            raise ValueError(f"Fold {fold_id} test manifest cohort mismatch: {te['cohort'].unique().tolist()}")
        if cohort in set(tr["cohort"]) or cohort in set(va["cohort"]):
            raise ValueError(f"Held-out cohort '{cohort}' leaked into source train/val on fold {fold_id}.")

        for name_a, da, name_b, db in [("train", tr, "val", va), ("train", tr, "test", te), ("val", va, "test", te)]:
            img_ov = set(da["image_key"]) & set(db["image_key"])
            grp_ov = set(da["patient_key"]) & set(db["patient_key"])
            if img_ov or grp_ov:
                raise ValueError(
                    f"Fold {fold_id} overlap ({name_a} vs {name_b}): "
                    f"images={len(img_ov)}, groups={len(grp_ov)}"
                )

        nat_tr, nat_meta = build_natural_source_train(folds_dir, fold_id, images_root=images_root)
        tr_in_nat = set(tr["image_key"]).issubset(set(nat_tr["image_key"]))
        if not tr_in_nat:
            raise ValueError(f"Fold {fold_id} quota train is not a subset of natural source_train.")

        # Label consistency check across quota train and natural source_train
        merged_lbl = tr[["image_key", "y_true"]].merge(
            nat_tr[["image_key", "y_true"]], on="image_key", suffixes=("_quota", "_nat")
        )
        if not (merged_lbl["y_true_quota"] == merged_lbl["y_true_nat"]).all():
            raise ValueError(f"Fold {fold_id} label discrepancy between train.csv and test/pool manifests.")

        if images_root is not None:
            verify_images_readable(te, check_decode=check_decode, max_checks=50 if check_decode else None)

        inventory_per_cohort[cohort] = int(len(te))
        fold_reports.append(
            {
                "fold_id": fold_id,
                "cohort": cohort,
                "quota_train_images": int(len(tr)),
                "quota_grade_counts": {int(k): int((tr["y_true"] == k).sum()) for k in range(5)},
                "natural_source_train_images": nat_meta["natural_source_train_images"],
                "natural_grade_counts": nat_meta["natural_grade_counts"],
                "source_val_images": int(len(va)),
                "target_images": int(len(te)),
                "target_g3_support": int((te["y_true"] == 3).sum()),
                "realized_natural_train_ratio": nat_meta["realized_train_ratio"],
                "quota_subset_of_natural": True,
                "sha256": {
                    "train": sha256_file(p_tr),
                    "val": sha256_file(p_va),
                    "test": sha256_file(p_te),
                },
            }
        )

    return {
        "total_target_evaluated_images": int(sum(inventory_per_cohort.values())),
        "target_counts_by_cohort": inventory_per_cohort,
        "folds": fold_reports,
    }


def validate_p0_cache(
    p0_dir: Path,
    folds: Sequence[int] = tuple(range(6)),
    variants: Sequence[str] = ("V2", "V3"),
    splits: Sequence[str] = ("source_train", "source_val", "target"),
    require_features: bool = False,
) -> Dict[str, Any]:
    """Verify P0 COMPLETE.json digests, finite probabilities, alignment, and optional z.npy arrays."""
    p0_dir = Path(p0_dir)
    lodo_dir = p0_dir / "lodo"
    if not lodo_dir.is_dir():
        raise FileNotFoundError(f"Missing P0 lodo/ directory at: {p0_dir}")

    checked_shards = 0
    ckpt_hashes: Dict[str, str] = {}
    for variant in variants:
        for fold in folds:
            cohort = CANONICAL_FOLD_TO_COHORT[fold]
            unit_dir = lodo_dir / f"{variant}_fold_{fold}_{cohort}"
            for split in splits:
                shard_dir = unit_dir / split
                comp_path = shard_dir / "COMPLETE.json"
                if not comp_path.is_file():
                    raise ValueError(f"Incomplete P0 shard (missing COMPLETE.json): {shard_dir}")
                meta = json.loads(comp_path.read_text(encoding="utf-8"))
                ident = meta.get("identity", {})
                ckpt_sha = ident.get("checkpoint_sha256")
                unit_key = f"{variant}_fold_{fold}"
                if ckpt_sha is not None:
                    if unit_key in ckpt_hashes and ckpt_hashes[unit_key] != ckpt_sha:
                        raise ValueError(f"Mixed checkpoint_sha256 in {unit_key}/{split}")
                    ckpt_hashes[unit_key] = ckpt_sha

                files_meta = meta.get("files", {})
                pred_path = shard_dir / "predictions.csv"
                if not pred_path.is_file():
                    raise FileNotFoundError(f"Missing predictions.csv in {shard_dir}")
                if "predictions.csv" in files_meta and sha256_file(pred_path) != files_meta["predictions.csv"]:
                    raise ValueError(f"Corrupted/tampered predictions.csv digest in {shard_dir}")

                df = pd.read_csv(pred_path)
                if df.empty or df["image_key"].duplicated().any():
                    raise ValueError(f"Empty or duplicate image_key in {pred_path}")
                p = df[list(PROB_COLS)].to_numpy(dtype=np.float64)
                if not np.isfinite(p).all() or (p < 0.0).any() or not np.allclose(p.sum(axis=1), 1.0, atol=1e-6, rtol=0.0):
                    raise ValueError(f"Invalid probabilities in {pred_path}")

                z_path = shard_dir / "z.npy"
                if require_features or z_path.is_file():
                    if not z_path.is_file():
                        raise FileNotFoundError(f"Missing z.npy in {shard_dir}")
                    if "z.npy" in files_meta and sha256_file(z_path) != files_meta["z.npy"]:
                        raise ValueError(f"Corrupted/tampered z.npy digest in {shard_dir}")
                    z = np.load(z_path, mmap_mode="r")
                    if z.ndim != 2 or z.shape[0] != len(df):
                        raise ValueError(f"Row mismatch z.npy {z.shape} vs predictions {len(df)} in {shard_dir}")
                    if not np.isfinite(np.asarray(z[: min(8, len(df))])).all():
                        raise ValueError(f"Non-finite values in {z_path}")

                checked_shards += 1

    return {"p0_dir": str(p0_dir), "checked_shards": checked_shards, "checkpoint_hashes": ckpt_hashes}


def save_predictions_table(df: pd.DataFrame, out_dir: Path, prefer_parquet: bool = True) -> Path:
    """Save canonical per-image predictions to compressed Parquet (with CSV fallback)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    missing = [c for c in REQUIRED_PREDICTION_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Predictions DataFrame missing required columns: {missing}")

    p = df[list(PROB_COLS)].to_numpy(dtype=np.float64)
    if not np.isfinite(p).all() or (p < 0.0).any() or not np.allclose(p.sum(axis=1), 1.0, atol=1e-5, rtol=0.0):
        raise ValueError("Refusing to write corrupted non-simplex probabilities.")

    if prefer_parquet:
        try:
            target = out_dir / "predictions.parquet"
            tmp = out_dir / f"predictions.{uuid4().hex[:8]}.parquet.tmp"
            df.to_parquet(tmp, index=False, compression="snappy")
            os.replace(tmp, target)
            return target
        except Exception:
            pass

    target = out_dir / "predictions.csv"
    tmp = out_dir / f"predictions.{uuid4().hex[:8]}.csv.tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, target)
    return target


def load_predictions_table(directory_or_file: Path) -> pd.DataFrame:
    p = Path(directory_or_file)
    if p.is_dir():
        pq = p / "predictions.parquet"
        cs = p / "predictions.csv"
        p = pq if pq.is_file() else cs
    if not p.is_file():
        raise FileNotFoundError(f"Predictions table not found: {directory_or_file}")
    if p.suffix == ".parquet":
        return pd.read_parquet(p)
    return pd.read_csv(p)


def save_feature_bundle(
    z: np.ndarray,
    image_keys: Sequence[str],
    out_dir: Path,
    dtype: str = "float32",
) -> Dict[str, Any]:
    """Write mmap-friendly z.npy + keyed z_manifest.json."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(z, dtype=dtype)
    if arr.ndim != 2 or arr.shape[0] != len(image_keys):
        raise ValueError(f"Feature shape {arr.shape} does not match {len(image_keys)} image keys.")
    if not np.isfinite(arr).all():
        raise ValueError("Refusing to cache non-finite latent feature array.")

    z_path = out_dir / "z.npy"
    tmp = out_dir / f"z.{uuid4().hex[:8]}.tmp.npy"
    np.save(tmp, arr)
    os.replace(tmp, z_path)

    manifest = {
        "schema_version": SCHEMA_VERSION,
        "feature_file": "z.npy",
        "dtype": str(arr.dtype),
        "shape": list(arr.shape),
        "sha256": sha256_file(z_path),
        "image_keys_sha256": object_hash(list(image_keys)),
    }
    atomic_write_json(out_dir / "z_manifest.json", manifest)
    return manifest

### Module `experiments.mlwa.reporting`

In [ ]:
"""Verified-only journal report generator, coverage audit, and measured-data publication plots."""


import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def _set_pub_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
            "font.size": 9.0,
            "axes.titlesize": 10.0,
            "axes.labelsize": 9.5,
            "xtick.labelsize": 8.5,
            "ytick.labelsize": 8.5,
            "legend.fontsize": 8.2,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "savefig.bbox": "tight",
            "savefig.pad_inches": 0.08,
        }
    )


def generate_journal_reports(
    output_dir: Path,
    p0_dir: Optional[Path] = None,
    e0_dir: Optional[Path] = None,
    e1_dir: Optional[Path] = None,
    journal_runs_dir: Optional[Path] = None,
) -> Dict[str, Any]:
    """Generate tables and measured figures ONLY from verified completed artifacts (never fabricate zeros)."""
    _set_pub_style()
    out_dir = Path(output_dir).resolve()
    tables_dir = out_dir / "tables"
    figs_dir = out_dir / "figures"
    tables_dir.mkdir(parents=True, exist_ok=True)
    figs_dir.mkdir(parents=True, exist_ok=True)

    coverage_rows: List[Dict[str, Any]] = []
    missing_runs: List[Dict[str, Any]] = []
    generated_tables: Dict[str, str] = {}
    generated_figures: Dict[str, Any] = {}
    input_digests: Dict[str, str] = {}

    # 1. Legacy P0 / E0 / E1 Baseline Verification (if supplied)
    if p0_dir is not None and (Path(p0_dir) / "G0_STATUS.json").is_file():
        p0_path = Path(p0_dir)
        g0_status = json.loads((p0_path / "G0_STATUS.json").read_text(encoding="utf-8"))
        pooled_status = json.loads((p0_path / "pooled_asset_status.json").read_text(encoding="utf-8"))
        coverage_rows.append(
            {"study": "P0_LODO_replay", "status": STATUS_DIAGNOSTIC_REPLAY, "source_dir": str(p0_path), "note": g0_status.get("status")}
        )
        coverage_rows.append(
            {"study": "P0_POOLED_5HEADS", "status": pooled_status.get("status", STATUS_PENDING_ASSETS), "source_dir": str(p0_path), "note": pooled_status.get("reason", "")}
        )
    else:
        missing_runs.append({"study": "P0_LODO_replay", "reason": "P0 directory not supplied or G0_STATUS.json missing"})

    e0_dm_path = (Path(e0_dir) / "lodo" / "decision_metrics.csv") if e0_dir else None
    if e0_dm_path and e0_dm_path.is_file():
        input_digests[str(e0_dm_path)] = sha256_file(e0_dm_path)
        e0_df = pd.read_csv(e0_dm_path)
        tgt = e0_df[e0_df["split"] == "target"].copy()
        tgt.to_csv(tables_dir / "legacy_e0_per_cohort_target.csv", index=False)
        generated_tables["legacy_e0_per_cohort_target"] = str(tables_dir / "legacy_e0_per_cohort_target.csv")
        coverage_rows.append({"study": "E0_legacy_decoders", "status": STATUS_COMPLETE, "source_dir": str(e0_dir), "note": "Single-seed (42); P0 G0=DIAGNOSTIC"})

        # Generate measured Pareto / Grade-3 vs Within-1 trade-off chart (CIBD vs native)
        sub = tgt[tgt["decoder"].isin(["native_replay", "cibd_selected", "argmax", "posterior_median", "rounded_expectation"])].copy()
        eq = sub.groupby(["variant", "decoder"], as_index=False)[["recall_g3", "within1", "qwk", "mae"]].mean()
        eq["recall_g3_pp"] = eq["recall_g3"] * 100.0
        eq["within1_pp"] = eq["within1"] * 100.0
        eq.to_csv(tables_dir / "legacy_e0_equal_cohort_summary.csv", index=False)
        generated_tables["legacy_e0_equal_cohort_summary"] = str(tables_dir / "legacy_e0_equal_cohort_summary.csv")

        fig, ax = plt.subplots(figsize=(6.6, 4.0))
        offsets = {
            ("V2", "argmax"): (-8, -11),
            ("V2", "native_replay"): (5, 3),
            ("V2", "posterior_median"): (5, 3),
            ("V2", "rounded_expectation"): (5, -9),
            ("V2", "cibd_selected"): (5, 3),
            ("V3", "argmax"): (5, 3),
            ("V3", "posterior_median"): (5, 4),
            ("V3", "cibd_selected"): (5, 3),
            ("V3", "native_replay"): (-6, 7),
            ("V3", "rounded_expectation"): (-12, -12),
        }
        for var, col in [("V2", "#0072B2"), ("V3", "#C0392B")]:
            v_eq = eq[eq["variant"] == var]
            ax.scatter(v_eq["within1_pp"], v_eq["recall_g3_pp"], color=col, s=60, label=f"{var} (archived replay)")
            for _, r in v_eq.iterrows():
                dec = str(r["decoder"])
                dx, dy = offsets.get((var, dec), (4, 3))
                ax.annotate(dec, (r["within1_pp"], r["recall_g3_pp"]), xytext=(dx, dy), textcoords="offset points", fontsize=7.5, color=col)
        ax.margins(x=0.08, y=0.12)
        ax.set_xlabel("Equal-cohort target Within-1 accuracy (%)")
        ax.set_ylabel("Equal-cohort target Grade 3 recall (%)")
        ax.set_title("Legacy E0 Operating Points (Single-Seed P0 Replay; G0=DIAGNOSTIC)")
        ax.grid(linestyle=":", alpha=0.35)
        ax.legend(frameon=True, facecolor="white")
        fig.savefig(figs_dir / "legacy_e0_recall_vs_within1.pdf")
        fig.savefig(figs_dir / "legacy_e0_recall_vs_within1.png", dpi=300)
        plt.close(fig)
        generated_figures["legacy_e0_recall_vs_within1"] = {
            "pdf": "figures/legacy_e0_recall_vs_within1.pdf",
            "data_csv": "tables/legacy_e0_equal_cohort_summary.csv",
            "aggregation": "equal_cohort_mean_6",
        }

        # Grade 3 destination flow chart separating 3->2 (still referable) from 3->0/1 (missed RDR)
        g3_src = tgt[(tgt["variant"] == "V3") & (tgt["decoder"].isin(["native_replay", "argmax", "posterior_median", "cibd_selected"]))].copy()
        flow_cols = [c for c in ["g3_to_0_count", "g3_to_1_count", "g3_to_2_count", "g3_to_3_count", "g3_to_4_count"] if c in g3_src.columns]
        if flow_cols:
            flow_sum = g3_src.groupby("decoder", as_index=False)[flow_cols].sum()
            flow_sum["g3_to_01_missed_rdr"] = flow_sum["g3_to_0_count"] + flow_sum["g3_to_1_count"]
            flow_sum.to_csv(tables_dir / "legacy_v3_g3_destinations_pooled.csv", index=False)
            generated_tables["legacy_v3_g3_destinations_pooled"] = str(tables_dir / "legacy_v3_g3_destinations_pooled.csv")
    else:
        missing_runs.append({"study": "E0_legacy_decoders", "reason": "decision_metrics.csv not found"})

    e1_eq_path = (Path(e1_dir) / "equal_cohort_means.csv") if e1_dir else None
    if e1_eq_path and e1_eq_path.is_file():
        input_digests[str(e1_eq_path)] = sha256_file(e1_eq_path)
        e1_eq = pd.read_csv(e1_eq_path)
        e1_eq.to_csv(tables_dir / "legacy_e1_equal_cohort_means.csv", index=False)
        generated_tables["legacy_e1_equal_cohort_means"] = str(tables_dir / "legacy_e1_equal_cohort_means.csv")
        coverage_rows.append({"study": "E1_legacy_probes", "status": STATUS_COMPLETE, "source_dir": str(e1_dir), "note": "s_v3=0.5409 > z_v3=0.5109; H3 gate closed"})
    else:
        missing_runs.append({"study": "E1_legacy_probes", "reason": "equal_cohort_means.csv not found"})

    # 2. Scan multi-seed GPU journal runs if present (never zero-fill unrun tables!)
    expected_journal_studies = [
        ("E2_2x2_factorial", 72),
        ("E2b_link_ablation", 18),
        ("E4_imbalance_baselines", 36),
        ("E3_brc", 0),
        ("E5_prior_adaptation", 0),
        ("E6_generality", 0),
    ]
    completed_units = []
    if journal_runs_dir is not None and Path(journal_runs_dir).is_dir():
        for comp in sorted(Path(journal_runs_dir).rglob("COMPLETE.json")):
            meta = json.loads(comp.read_text(encoding="utf-8"))
            if meta.get("status") == STATUS_COMPLETE:
                completed_units.append({"path": str(comp.parent), **meta.get("identity", {})})

    for st_name, req_n in expected_journal_studies:
        if st_name == "E3_brc":
            coverage_rows.append({"study": st_name, "status": STATUS_SKIPPED_GATE_NOT_MET, "source_dir": str(journal_runs_dir or ""), "note": "Gated by H3"})
        elif req_n > 0:
            n_done = len([u for u in completed_units if u.get("experiment", "").startswith(st_name.split("_")[0])])
            st_status = STATUS_COMPLETE if n_done >= req_n else ("PARTIAL" if n_done > 0 else "NOT_YET_RUN")
            coverage_rows.append({"study": st_name, "status": st_status, "source_dir": str(journal_runs_dir or ""), "note": f"{n_done}/{req_n} verified units"})
            if n_done < req_n:
                missing_runs.append({"study": st_name, "reason": f"Only {n_done}/{req_n} verified GPU training units present (no fabrication)."})

    cov_df = pd.DataFrame(coverage_rows)
    cov_df.to_csv(tables_dir / "experiment_coverage.csv", index=False)
    miss_df = pd.DataFrame(missing_runs)
    miss_df.to_csv(tables_dir / "missing_runs.csv", index=False)

    manifest = {
        "schema": "mlwa_report_manifest_v1",
        "output_dir": str(out_dir),
        "input_file_sha256": input_digests,
        "coverage_table": "tables/experiment_coverage.csv",
        "missing_runs_table": "tables/missing_runs.csv",
        "generated_tables": generated_tables,
        "generated_figures": generated_figures,
    }
    atomic_write_json(out_dir / "report_manifest.json", manifest)
    return manifest

## Experiment Workflow & Entry Points

In [ ]:
import argparse
import json
from typing import Optional, Sequence


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    paths = resolve_runtime_paths()
    res_root = paths['repo_root'] / 'result'
    parser = argparse.ArgumentParser(description='15: Generate verified-only MLWA journal tables, plots, and coverage audit.')
    parser.add_argument('--p0-dir', type=Path, default=paths['p0_run'])
    parser.add_argument('--e0-dir', type=Path, default=res_root / 'p0_fp32_main_lodo_v1_e0_resume_v2')
    parser.add_argument('--e1-dir', type=Path, default=res_root / 'p0_fp32_main_lodo_v1_e1_resume_v2')
    parser.add_argument('--journal-runs-dir', type=Path, default=None)
    parser.add_argument('--output-dir', type=Path, default=paths['outputs'] / 'mlwa_journal_reports')
    parser.add_argument('--smoke', action='store_true', help='Generate coverage-only report into output-dir with no fabrication.')
    return parser.parse_args(argv)


def main(argv: Optional[Sequence[str]] = None) -> int:
    args = parse_args(argv)
    if args.smoke:
        manifest = generate_journal_reports(
            output_dir=args.output_dir,
            p0_dir=None,
            e0_dir=None,
            e1_dir=None,
            journal_runs_dir=None,
        )
    else:
        manifest = generate_journal_reports(
            output_dir=args.output_dir,
            p0_dir=args.p0_dir,
            e0_dir=args.e0_dir,
            e1_dir=args.e1_dir,
            journal_runs_dir=args.journal_runs_dir,
        )
    print(json.dumps({
        'output_dir': manifest['output_dir'],
        'generated_tables': list(manifest['generated_tables'].keys()),
        'generated_figures': list(manifest['generated_figures'].keys()),
        'coverage_table': manifest['coverage_table'],
        'missing_runs_table': manifest['missing_runs_table'],
    }, indent=2))
    return 0

In [ ]:
if __name__ == '__main__' and _running_in_notebook():
    _paths = resolve_runtime_paths(_REPO)
    _res_root = _paths['repo_root'] / 'result'
    _out = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else (_paths['outputs'] / 'mlwa_journal_reports')
    _nb_manifest = generate_journal_reports(
        output_dir=_out,
        p0_dir=None if SMOKE else (P0_DIR if P0_DIR is not None else _paths['p0_run']),
        e0_dir=None if SMOKE else (E0_DIR if E0_DIR is not None else (_res_root / 'p0_fp32_main_lodo_v1_e0_resume_v2')),
        e1_dir=None if SMOKE else (E1_DIR if E1_DIR is not None else (_res_root / 'p0_fp32_main_lodo_v1_e1_resume_v2')),
        journal_runs_dir=None if SMOKE else JOURNAL_RUNS_DIR,
    )
    print(json.dumps({'output_dir': _nb_manifest['output_dir'], 'tables': list(_nb_manifest['generated_tables'].keys())}, indent=2))

In [ ]:
if __name__ == '__main__' and not _running_in_notebook():
    raise SystemExit(main())